# 04. LLM Wiki: 새 자료 D6가 기존 지식을 바꾸는 과정

이 예시는 [Karpathy의 LLM Wiki 원문](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)의 세 층 **raw / wiki / schema(AGENTS.md)**, 세 작업 **ingest / query / lint**를 결제 프로젝트에 적용합니다. `wiki_vault/`는 D1–D5가 반영된 초기 상태입니다. `index.md`는 주제별 지도, `log.md`는 시간순 기록입니다.

기본 실행은 미리 작성한 D6 계획(`fixture`)을 사용해 API 없이 전후 diff를 재현합니다. 아래 선택 셀에서 `llm` 모드로 바꾸면 실제 에이전트가 새 계획을 제안합니다. 둘을 혼동해 설명하지 않습니다.

`wiki_vault/`는 D1–D5 초기 템플릿입니다. 실행하며 바뀌는 파일은 `runs/wiki_notebook_.../vault/`에 저장됩니다. 같은 커널에서 첫 셀을 다시 실행해도 기존 작업 위키를 재사용합니다. 커널을 새로 시작하면 새 복사본이 만들어집니다.


In [ ]:
from pathlib import Path
from datetime import datetime
import shutil, sys
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from session_demo.wiki_cli import plan_ingest, preview, apply_plan, answer_extractively, lint, WIKILINK

# 같은 커널에서 첫 셀을 다시 눌러도 이미 작업하던 위키를 유지합니다.
previous_vault = globals().get("VAULT")
if (
    isinstance(previous_vault, Path)
    and previous_vault.name == "vault"
    and previous_vault.is_dir()
    and previous_vault.parent.parent == ROOT / "runs"
    and previous_vault.parent.name.startswith("wiki_notebook_")
):
    VAULT = previous_vault
    RUN = VAULT.parent
    print("기존 실행의 위키 재사용:", VAULT)
else:
    RUN = ROOT / "runs" / f"wiki_notebook_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
    VAULT = RUN / "vault"
    RUN.mkdir(parents=True)
    shutil.copytree(ROOT / "wiki_vault", VAULT)
    print("새 실행의 위키:", VAULT)


## 1. 세 층과 초기 상태

`raw/`는 수정하지 않는 원문, `wiki/`는 새 자료에 맞춰 계속 고치는 지식 페이지, `AGENTS.md`는 두 층을 다루는 규칙입니다. 아래에서 **규칙 파일 전체**를 읽고 ingest·query·lint가 각각 무엇을 해야 하는지 확인합니다. Karpathy의 글은 패턴 제안이며, 이 파일의 문서 ID·검토 절차는 시연 자료에 맞춘 구체화입니다. `Obsidian → Open folder as vault`에서 위 출력 경로를 열면 파일과 링크를 화면으로 볼 수 있습니다.


In [ ]:
print("[구조]")
for path in sorted(VAULT.rglob("*.md")):
    print("-", path.relative_to(VAULT))
print("\n[AGENTS.md: 실제 LLM 작업에 제공되는 운영 규칙 전체]")
display(Markdown((VAULT / "AGENTS.md").read_text(encoding="utf-8")))
print("\n[D6 이전 출시 일정]\n", (VAULT / "wiki" / "출시 일정.md").read_text(encoding="utf-8"))


## 2. Ingest: D6를 넣기 전에 변경 계획과 diff 보기

새 결정 D6는 출시 목표일을 11월 3일에서 11월 10일로 바꿉니다. 처음에는 작성해 둔 **시연용 계획**을 로드합니다. 계획은 원자료와 기존 위키의 해시를 기록해, 검토 후 파일이 바뀌었다면 적용을 거부합니다. 원문은 diff를 확인한 뒤 `raw/`에 보존됩니다.


In [ ]:
source = ROOT / "data" / "project" / "D6_decision.md"
plan_path = RUN / "D6_plan.json"
d6_raw = VAULT / "raw" / "D6_decision.md"
if d6_raw.is_file():
    print("이 작업 위키에는 D6가 이미 적용되어 있습니다:", VAULT)
else:
    plan = plan_ingest(VAULT, source, mode="fixture", plan_path=plan_path)
    print(preview(VAULT, plan))


In [ ]:
plan_path = RUN / "D6_plan.json"
d6_raw = VAULT / "raw" / "D6_decision.md"
if d6_raw.is_file():
    print("D6 적용을 건너뛰었습니다. 이미 이 작업 위키에 있습니다:", VAULT)
else:
    if not plan_path.is_file():
        raise RuntimeError("현재 위키의 D6 계획이 없습니다. 바로 위 계획·diff 셀을 먼저 실행하세요.")
    changed = apply_plan(VAULT, plan_path)
    print("적용 파일:", changed)
print("\n[D6 이후 출시 일정]\n", (VAULT / "wiki" / "출시 일정.md").read_text(encoding="utf-8"))
print("\n[D6 이후 프로그램 개요]\n", (VAULT / "wiki" / "결제 안정화 프로그램.md").read_text(encoding="utf-8"))
print("\n[D6 이후 보안 검토]\n", (VAULT / "wiki" / "보안 검토.md").read_text(encoding="utf-8"))


## 3. Query: 바뀐 현재 사실과 원문 근거 찾기

아래 오프라인 답은 규칙 기반 추출이며 LLM 합성 답이 아닙니다. 관련 위키 페이지를 검색하고, `출시 일정.md`의 현재 날짜와 변경 이유를 표시합니다. 앞의 Ingest 적용 셀을 건너뛴 경우 이 셀은 **준비된 시연용 D6 계획만 자동으로 적용**하고 그 사실을 출력합니다. 실제 LLM 계획은 자동 적용하지 않습니다. LLM 질의는 뒤의 선택 셀에서 할 수 있습니다.


In [ ]:
question = "현재 출시 목표일은 언제이며 왜 바뀌었나?"
d6_raw = VAULT / "raw" / "D6_decision.md"
if not d6_raw.is_file():
    # 교육용 고정 fixture만 자동 적용합니다. 실제 LLM 계획은 여기서 적용하지 않습니다.
    fixture_source = ROOT / "data" / "project" / "D6_decision.md"
    fixture_plan_path = RUN / "D6_query_fixture_plan.json"
    fixture_plan = plan_ingest(VAULT, fixture_source, mode="fixture", plan_path=fixture_plan_path)
    changed = apply_plan(VAULT, fixture_plan_path)
    print("[시연용 D6 자료 자동 준비] Ingest 적용 셀이 생략되어 고정 예시 계획을 적용했습니다.")
    print("적용 파일:", changed)
print("현재 작업 위키:", VAULT)
print(answer_extractively(VAULT, question))
print("\n[D6 원문]\n", d6_raw.read_text(encoding="utf-8"))


## 4. Lint: 깨진 링크·출처·낡은 현재 사실 찾기

첫 번째는 갱신된 실제 위키를 점검합니다. 두 번째는 별도 복사본에 낡은 날짜를 일부러 넣어 **lint가 어떤 문제를 찾는지** 보여줍니다. 원래 위키는 수정하지 않습니다.


In [ ]:
print("[정상 위키 점검]", lint(VAULT) or "발견 항목 없음")
broken = RUN / f"broken_copy_{datetime.now().strftime('%H%M%S_%f')}"
shutil.copytree(VAULT, broken)
schedule = broken / "wiki" / "출시 일정.md"
schedule.write_text(
    schedule.read_text(encoding="utf-8").replace("현재 출시 목표일: 2025-11-10", "현재 출시 목표일: 2025-11-03"),
    encoding="utf-8",
)
print("\n[의도적으로 낡은 날짜를 넣은 복사본]")
for issue in lint(broken):
    print("-", issue)


## 5. 위키 링크 그래프 보기

이 선은 `[[페이지 링크]]`가 있는 두 페이지를 한 번만 연결한 것입니다. 화면에서는 링크 방향을 생략합니다. Neo4j의 `LEADS`, `PART_OF`, `REVIEWS`처럼 타입과 방향이 확인된 사실 관계는 아닙니다. Obsidian 그래프 뷰에서도 같은 차이를 짚습니다.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import textwrap
from session_demo.plot_fonts import apply_korean_font

# 위키 페이지 사이에 링크가 있으면 한 선으로 표시합니다. 방향은 여기서 생략합니다.
link_graph = nx.Graph()
for page in (VAULT / "wiki").glob("*.md"):
    link_graph.add_node(page.stem)
    for target in WIKILINK.findall(page.read_text(encoding="utf-8")):
        link_graph.add_edge(page.stem, target)
fig, ax = plt.subplots(figsize=(10, 6))
pos = nx.spring_layout(link_graph, seed=5, k=1.2)
nx.draw_networkx_nodes(link_graph, pos, ax=ax, node_color="#f9e0bf", node_size=4200, edgecolors="#c99458")
nx.draw_networkx_edges(link_graph, pos, ax=ax, edge_color="#b3916c", width=1.7)
labels = {name: textwrap.fill(name, width=8, break_long_words=False) for name in link_graph}
nx.draw_networkx_labels(link_graph, pos, labels=labels, ax=ax, font_size=10)
apply_korean_font(ax)
ax.margins(0.24)
ax.axis("off")
fig.tight_layout()
display(fig)
plt.close(fig)


## 6. 실제 LLM 에이전트로 계획·질의·의미 lint 실행 (선택)

LLM 의미 lint는 **검토할 제안**입니다. 같은 사실의 반복은 모순이 아니며, 링크가 들어오는 페이지를 고립 페이지라고 부르면 그래프·기계적 lint와 다시 대조합니다.

키가 준비됐다면 새 복사본에서 `llm` 계획을 만들고 diff를 확인합니다. 이 셀은 **계획만 생성하고 적용하지 않습니다.** `apply_plan`에 같은 plan 파일을 넣어야 합니다. 질의·의미 lint도 각각 유료 호출입니다. 모델 계획이 출처 검증에 실패하면 후보 JSON을 저장하고 최대 두 번 다시 제안받습니다(추가 API 호출). 재제안이 일부 파일만 고치면 앞선 계획과 합쳐 전체를 다시 검증합니다. 끝까지 검증을 통과하지 못하면 적용하지 않으며, 질의·lint는 별도로 진행합니다.


In [ ]:
RUN_LLM_INGEST = False
RUN_LLM_QUERY = False
RUN_LLM_LINT = False

if RUN_LLM_INGEST:
    print("\n[LLM ingest 변경 계획]")
    live_vault = RUN / f"live_agent_vault_{datetime.now().strftime('%H%M%S_%f')}"
    shutil.copytree(ROOT / "wiki_vault", live_vault)
    live_plan_path = RUN / "D6_live_plan.json"
    try:
        live_plan = plan_ingest(live_vault, source, mode="llm", plan_path=live_plan_path)
    except ValueError as exc:
        print("제안이 검증에서 거부됐습니다:", exc)
        print("적용하지 않았습니다. 저장된 candidate JSON과 원문을 대조하세요.")
    else:
        print(preview(live_vault, live_plan))
        print("계획 저장:", live_plan_path)
        print("검토 후 적용 예: apply_plan(live_vault, live_plan_path)")
if RUN_LLM_QUERY:
    from session_demo.wiki_cli import answer_with_llm
    print("\n[LLM query: D6 반영 위키]")
    print(answer_with_llm(VAULT, question))
if RUN_LLM_LINT:
    from session_demo.wiki_cli import qualitative_lint
    print("\n[LLM 의미 lint: D6 반영 위키]")
    print(qualitative_lint(VAULT))
    print("기계적 lint 대조:", lint(VAULT) or "발견 항목 없음")
if not any((RUN_LLM_INGEST, RUN_LLM_QUERY, RUN_LLM_LINT)):
    print("LLM 호출을 건너뛰었습니다. 위의 fixture 전후 diff·검색·기계적 lint는 실행되었습니다.")


**세션 마무리:** GraphRAG에서는 질문 시 관계와 원문 근거를 회수했습니다. LLM Wiki에서는 D6가 들어오자 이미 존재하던 일정·프로그램·보안 페이지를 함께 고쳤습니다. 두 방식 모두 출처와 사람의 검토가 중요합니다.
